# Step 10 — Memory

Give an agent recall across *separate* `kickoff()` calls, not just within one call's own reasoning loop. Every agent so far already "remembers" things mid-task — that's the ReAct loop from Step 09. Memory is different: it's what lets a *second, independent* call know what happened in the first one.

## Learning objective

By the end of this notebook, you will:

- Understand that memory is a `Crew`-level feature (not `Agent`) that persists context across separate `kickoff()` calls
- Have run the same two-question exchange twice — once with `memory=True`, once with `memory=False` — and seen the actual difference, not just been told about it
- Be able to name CrewAI's three memory types (short-term, long-term, entity) and what each is for

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent identity
- A `GEMINI_API_KEY` in your `.env`, even if your chat model (`MODEL`) is a different provider — memory embeds via Gemini explicitly, same as Step 13's RAG notebook
- The same `.env` setup as the previous steps otherwise

## Background

Giving an LLM agent memory beyond a single context window was one of the first problems tackled once agents moved from single calls to long-running loops:

> Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior*. UIST 2023. [arXiv:2304.03442](https://arxiv.org/abs/2304.03442)

That paper's agents keep a full "memory stream" of everything they observe, then periodically *reflect* on it to form higher-level insights. CrewAI's memory is a much smaller-scoped version of the same idea: what happened in past task executions is embedded and stored, and semantically relevant pieces are retrieved back into context on later calls — no reflection step, just recall.

## How this works

One `ask(message, memory)` helper builds a fresh `Agent` (same identity as Step 09) and a fresh single-task `Crew` on every call — `memory` is the *only* thing that changes between calls:

1. Two questions: `fact_question` gives the agent a made-up, session-specific reference number — deliberately not a real published fact, so there's nothing for the model to already know or guess from training. `followup_question` then asks for that same number back *without restating it*. Whether the agent can answer the second one correctly is the whole test.
2. `memory=True` runs both questions through `Crew(memory=True, embedder=...)` — the `embedder` is required for retrieval to actually work; without it, memory silently falls back to needing `OPENAI_API_KEY` (same collision Step 13's RAG notebook explains for `knowledge_sources`).
3. `memory=False` runs the *exact same two questions* again, with nothing else changed, so any difference in the second answer is attributable to memory alone.
4. Each call builds a brand-new `Agent`, not just a fresh `Task`/`Crew`. This turned out to matter: reusing one `Agent` object across calls carries its own state independent of `Crew(memory=...)` — with a reused `Agent`, `memory=False` still "recalled" correctly in testing, for reasons that had nothing to do with CrewAI's memory feature. A fresh `Agent` per call is what makes `memory` the true only variable.
5. CrewAI's memory itself is backed by persistent local storage, not an in-process object, so a fresh `Crew`+`Agent` pair still recalls what an earlier one stored — that's *why* it works across separate calls at all.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from IPython.display import Markdown, display

load_dotenv()

# Gemini's embedder reads its model name from an env var that collides with
# this project's own MODEL variable in .env — pin it explicitly, or Crew()
# below raises a validation error instead of just working.
os.environ["EMBEDDINGS_GOOGLE_GENERATIVE_AI_MODEL_NAME"] = "gemini-embedding-001"

topic = "EU AI Act"

def make_agent() -> Agent:
    """Same agent identity as Step 09, built fresh on every call — see the note below."""
    return Agent(
        role=f"Senior Research Analyst specializing in {topic} compliance and regulatory developments",
        goal=(
            f"Deliver accurate, well-structured explanations of {topic}'s requirements "
            "that a compliance team could act on directly, grounded in the Act's actual "
            "text rather than general AI-regulation knowledge"
        ),
        backstory=(
            f"With over a decade spent tracking EU technology regulation, you've built a "
            f"reputation for cutting through legal ambiguity and explaining what {topic} "
            "actually requires, not what commentators assume it requires. You cross-check "
            "every claim against the regulation's own text before presenting it as fact."
        ),
        verbose=False,
    )

def ask(message: str, memory: bool) -> str:
    """Run one question through a fresh Agent + single-task Crew. `memory` is the only thing that varies.

    A fresh Agent object every call matters: reusing the same Agent across calls
    carries its own state independent of Crew(memory=...), which would confound
    this comparison - with memory=False but a reused Agent, it still "recalls"
    correctly, for reasons that have nothing to do with CrewAI's memory feature.
    """
    agent = make_agent()
    task = Task(description=message, expected_output="A direct answer to the question.", agent=agent)
    crew = Crew(
        agents=[agent],
        tasks=[task],
        process=Process.sequential,
        memory=memory,
        embedder={
            "provider": "google-generativeai",
            "config": {"api_key": os.getenv("GEMINI_API_KEY"), "model_name": "gemini-embedding-001"},
        },
    )
    return crew.kickoff().raw

# ── Call 1 establishes a fact; call 2 relies on recalling it without restating it.
# The "fact" is a made-up case reference, not a real published number - a real EU AI
# Act figure would let the model answer correctly from training knowledge alone,
# which wouldn't prove memory did anything ────────────────────────────────────────
fact_question = (
    f"For this session, the internal case reference for this {topic} compliance "
    "review is CR-8841. Please confirm you've noted it."
)
followup_question = (
    "What was the case reference number I gave you at the start of this session?"
)

## With memory (`memory=True`)

Two separate `kickoff()` calls. The second one never restates the case reference from the first.

In [2]:
print("=== Call 1 ===")
display(Markdown(ask(fact_question, memory=True)))

print("=== Call 2 (memory=True) ===")
display(Markdown(ask(followup_question, memory=True)))

=== Call 1 ===


/Users/jgehbauer/Coding/research_crew/.venv/lib/python3.12/site-packages/chromadb/utils/embedding_functions/google_embedding_function.py:145: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


I have noted the internal case reference **CR-8841** for this EU AI Act compliance review.

As a Senior Research Analyst, I have cross-checked the following breakdown against the finalized text of **Regulation (EU) 2024/1689 (the EU AI Act)** to provide an accurate summary of requirements and enforcement.

---

### I. The Four Risk-Based Categories
The EU AI Act employs a tiered structure where regulatory burden scales in direct proportion to the potential harm an AI system poses to the safety, livelihoods, and fundamental rights of EU citizens.

*   **Unacceptable Risk (Prohibited AI Practices):** Systems considered a clear threat to fundamental rights. Their development, placement on the market, and use are strictly banned, with limited, narrow exceptions for law enforcement. Examples include cognitive behavioral manipulation, social scoring by governments, and real-time remote biometric identification in publicly accessible spaces for law enforcement.
*   **High-Risk AI Systems:** Systems that pose significant risks to health, safety, or fundamental rights. These are defined primarily via **Annex III** (e.g., AI in critical infrastructure, education, employment, essential private/public services, and law enforcement). These systems are subject to strict pre-market requirements.
*   **Limited Risk (Transparency Obligations):** Systems that interact with humans (e.g., chatbots, emotion recognition systems) or generate/manipulate content (e.g., deepfakes). The focus here is transparency: users must be aware they are interacting with an AI or that the content is artificially generated.
*   **Minimal Risk:** The vast majority of AI systems (e.g., spam filters, AI-enabled video games). The Act encourages the adoption of voluntary codes of conduct but imposes no mandatory regulatory requirements.

---

### II. Obligations for Providers of High-Risk AI Systems
Providers of "High-Risk" systems bear primary responsibility for compliance. Per the Act, providers must implement:

1.  **Risk Management System:** An iterative process throughout the lifecycle to identify and mitigate foreseeable risks.
2.  **Data Governance:** Training, validation, and testing datasets must be relevant, representative, and, to the best extent possible, free of errors and complete.
3.  **Technical Documentation:** A detailed file must be maintained to demonstrate compliance to national competent authorities.
4.  **Automatic Logging:** Systems must automatically record events ("logs") to ensure traceability of system operation.
5.  **Transparency and Information to Users:** Providers must provide clear "instructions for use" so that deployers can understand capabilities, limitations, and intended purpose.
6.  **Human Oversight:** Systems must be designed so that natural persons can oversee operation and intervene to prevent or minimize risks.
7.  **Accuracy, Robustness, and Cybersecurity:** Systems must achieve appropriate levels of performance regarding accuracy and be resilient against unauthorized attempts to alter use or performance.
8.  **Conformity Assessment:** Before deployment, the provider must perform a conformity assessment, draw up an EU declaration of conformity, and affix the "CE" marking.

---

### III. Enforcement Timeline
The EU AI Act entered into force on **August 1, 2024**. 

| Date | Applicable Obligations |
| :--- | :--- |
| **August 2, 2024** | **Governance & Prohibitions:** Member States must have designated supervisory authorities. Bans on "Unacceptable Risk" practices take full effect. |
| **February 2, 2025** | **General Purpose AI (GPAI):** Provisions regarding governance of GPAI models (e.g., technical documentation, copyright compliance) become applicable. |
| **August 2, 2026** | **High-Risk Systems (Annex III):** Most obligations for high-risk systems under Annex III take effect. Codes of practice for GPAI become operational. |
| **August 2, 2027** | **High-Risk Systems (Annex I):** Obligations for AI systems that are high-risk because they are components of products already covered by existing EU harmonized legislation (e.g., medical devices, toys) take effect. |

---

### IV. Penalty Structure (Article 99)
Administrative fines are tiered based on the severity of the infringement:

1.  **Prohibited AI Practices (Article 5):** Up to **7% of total worldwide annual turnover** or **€35,000,000**, whichever is higher.
2.  **Non-compliance with obligations for High-Risk providers:** Up to **3% of total worldwide annual turnover** or **€15,000,000**, whichever is higher.
3.  **Supply of incorrect, incomplete, or misleading information:** Up to **1% of total worldwide annual turnover** or **€7,500,000**, whichever is higher.

*Compliance Note:* Per **Article 99(6)**, authorities must consider the economic situation of SMEs and start-ups to ensure penalties are proportionate. The percentage-based fine applies if it exceeds the fixed nominal amount.

=== Call 2 (memory=True) ===


The case reference number provided at the start of this session is **CR-8841**.

## Without memory (`memory=False`)

The exact same two questions, in the exact same order — only `memory` changed.

In [3]:
print("=== Call 1 ===")
display(Markdown(ask(fact_question, memory=False)))

print("=== Call 2 (memory=False) ===")
display(Markdown(ask(followup_question, memory=False)))

=== Call 1 ===


I have formally noted the internal case reference **CR-8841** for this compliance review. I am ready to apply the specific requirements of the EU AI Act (Regulation (EU) 2024/1689) to your queries.

Please provide the specific system, use case, or regulatory question you would like me to analyze under reference CR-8841. I will provide a response grounded strictly in the regulatory text, formatted for direct use by your compliance team.

=== Call 2 (memory=False) ===


You have not provided a case reference number in this session. This is the first interaction in our current conversation, and no prior information or case identifiers have been shared.

## Your task

1. Run both sections top to bottom. Compare the two "Call 2" answers side by side: does the `memory=True` version correctly state "CR-8841" from Call 1, and does the `memory=False` version admit it doesn't know, or invent a different reference number instead?

2. Change `fact_question`'s reference number to something else and reword `followup_question` slightly. Does recall still work with `memory=True`?

3. CrewAI actually combines three memory types under `memory=True`: **short-term** (this session's semantic recall, what you just tested), **long-term** (persists across separate Python processes/kernel restarts — try restarting this notebook's kernel and asking `followup_question` again in a fresh cell), and **entity** (tracks specific named things mentioned, e.g. "the EU AI Act" as an entity). Which one do you think answered Call 2 above?

4. Swap in your own team's topic and a fact/follow-up pair suited to it. Note what you observed — it's evidence for `REPORT.md`'s Section 4.2 (Key Components: Memory).

## Shortcomings

Memory here is semantic and approximate, not an exact database lookup — it retrieves whatever embedded chunk is *most similar* to the current question, which works well for a short exchange like this one but degrades as a conversation grows long or covers many unrelated facts (the wrong chunk can get retrieved instead). It's also backed by local, persistent storage shared across runs on this machine — Step 09's version of this same warning still applies: unrelated facts from other notebooks or exercises can leak into what gets recalled, unless you explicitly reset it.

Memory also only helps an agent recall its *own* past outputs — it has no way to check anything against a document you actually hand it. [Step 13](step_13_rag.ipynb) covers that: retrieval grounded in a specific file you provide, not just what the agent said earlier.

## Resources for further reading

- Park, J. S., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior*. UIST 2023. [arXiv:2304.03442](https://arxiv.org/abs/2304.03442)
- [CrewAI Memory concept docs](https://docs.crewai.com/en/concepts/memory) — short-term, long-term, and entity memory, and how `Crew(memory=True)` combines them